#Cleaning (NO encoding)
This notebook handles missing values, outliers, and adds `customer_id`, then saves the clean data. **Categories stay as words — no encoding here.**

### Import tools and load the data

In [1]:
import numpy as np
import pandas as pd
import seaborn as sb
import matplotlib.pyplot as plt

file_path = '../credit_risk_dataset.csv'
df = pd.read_csv(file_path)

### First look
See the rows, size, where nulls are, and the number summary.

In [2]:
print(df.head())
print('Shape:', df.shape)
print(df.isnull().sum())
print(df.describe())

   person_age  person_income person_home_ownership  person_emp_length  \
0          22          59000                  RENT              123.0   
1          21           9600                   OWN                5.0   
2          25           9600              MORTGAGE                1.0   
3          23          65500                  RENT                4.0   
4          24          54400                  RENT                8.0   

  loan_intent loan_grade  loan_amnt  loan_int_rate  loan_status  \
0    PERSONAL          D      35000          16.02            1   
1   EDUCATION          B       1000          11.14            0   
2     MEDICAL          C       5500          12.87            1   
3     MEDICAL          C      35000          15.23            1   
4     MEDICAL          C      35000          14.27            1   

   loan_percent_income cb_person_default_on_file  cb_person_cred_hist_length  
0                 0.59                         Y                           3  


### Step 1 — Handle missing values
`person_emp_length` → median. `loan_int_rate` → median of its own loan grade (rate depends on grade).

In [3]:
df['person_emp_length'] = df['person_emp_length'].fillna(df['person_emp_length'].median())

df['loan_int_rate'] = df.groupby('loan_grade')['loan_int_rate'].transform(
    lambda s: s.fillna(s.median())
)

print(df.isnull().sum())

person_age                    0
person_income                 0
person_home_ownership         0
person_emp_length             0
loan_intent                   0
loan_grade                    0
loan_amnt                     0
loan_int_rate                 0
loan_status                   0
loan_percent_income           0
cb_person_default_on_file     0
cb_person_cred_hist_length    0
dtype: int64


### Step 2 — Handle outliers
**(a)** Drop impossible values (age > 100, employment > 60 yrs = data errors). **(b)** Cap extreme income with the IQR fence `Q3 + 1.5*IQR` — cap, don't delete.

In [4]:
# (a) remove impossible values
df = df[df['person_age'] <= 100]
df = df[df['person_emp_length'] <= 60]

# (b) cap extreme income using the IQR fence
Q1 = df['person_income'].quantile(0.25)
Q3 = df['person_income'].quantile(0.75)
IQR = Q3 - Q1
upper_fence = Q3 + 1.5 * IQR
df['person_income'] = df['person_income'].clip(upper=upper_fence)

df = df.reset_index(drop=True)
print('Rows:', df.shape[0], '| Max income:', df['person_income'].max())

Rows: 32574 | Max income: 140250


### Step 3 — Create customer_id
Unique ID like `CUST_00001` as the first column.

In [5]:
df.insert(0, 'customer_id', ['CUST_' + str(i + 1).zfill(5) for i in range(len(df))])
print(df.head())

  customer_id  person_age  person_income person_home_ownership  \
0  CUST_00001          21           9600                   OWN   
1  CUST_00002          25           9600              MORTGAGE   
2  CUST_00003          23          65500                  RENT   
3  CUST_00004          24          54400                  RENT   
4  CUST_00005          21           9900                   OWN   

   person_emp_length loan_intent loan_grade  loan_amnt  loan_int_rate  \
0                5.0   EDUCATION          B       1000          11.14   
1                1.0     MEDICAL          C       5500          12.87   
2                4.0     MEDICAL          C      35000          15.23   
3                8.0     MEDICAL          C      35000          14.27   
4                2.0     VENTURE          A       2500           7.14   

   loan_status  loan_percent_income cb_person_default_on_file  \
0            0                 0.10                         N   
1            1                 0.5

### Save the clean file
Nulls + outliers handled, `customer_id` added, categories still as words.

In [6]:
df.to_csv('./credit_risk_clean.csv', index=False)
print('Saved clean file:', df.shape)

Saved clean file: (32574, 13)
